In [ ]:
import copy
from heapq import heappush, heappop


n = 3


rows = [ 1, 0, -1, 0 ]
cols = [ 0, -1, 0, 1 ]


class priorityQueue:
    def __init__(self):
        self.heap = []


    def push(self, key):
        heappush(self.heap, key)


    def pop(self):
        return heappop(self.heap)


    def empty(self):
        if not self.heap:
            return True
        else:
            return False


class nodes:
    def __init__(self, parent, mats, empty_tile_posi, costs, levels):
        self.parent = parent
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.costs = costs
        self.levels = levels

    def __lt__(self, nxt):
        return self.costs < nxt.costs

def calculateCosts(mats, final) -> int:
    count = 0
    for i in range(n):
        for j in range(n):
            if (mats[i][j]) and (mats[i][j] != final[i][j]):
                count += 1
    return count

def newNodes(mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final) -> nodes:

    new_mats = copy.deepcopy(mats)


    x1 = empty_tile_posi[0]
    y1 = empty_tile_posi[1]
    x2 = new_empty_tile_posi[0]
    y2 = new_empty_tile_posi[1]

    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]


    costs = calculateCosts(new_mats, final)

    new_nodes = nodes(parent, new_mats, new_empty_tile_posi, costs, levels)
    return new_nodes


def printMatsrix(mats):
    for i in range(n):
        for j in range(n):
            print("%d " % (mats[i][j]), end = " ")
        print()


def isSafe(x, y):
    return x >= 0 and x < n and y >= 0 and y < n


def printPath(root):
    if root == None:
        return

    printPath(root.parent)
    printMatsrix(root.mats)
    print()

def solve(initial, empty_tile_posi, final):
    pq = priorityQueue()


    costs = calculateCosts(initial, final)
    root = nodes(None, initial, empty_tile_posi, costs, 0)


    pq.push(root)

    while not pq.empty():
        minimum = pq.pop()


        if minimum.costs == 0:
            printPath(minimum)
            return


        for i in range(n):
            new_tile_posi = [
                minimum.empty_tile_posi[0] + rows[i],
                minimum.empty_tile_posi[1] + cols[i]
            ]

            if isSafe(new_tile_posi[0], new_tile_posi[1]):

                child = newNodes(
                    minimum.mats,
                    minimum.empty_tile_posi,
                    new_tile_posi,
                    minimum.levels + 1,
                    minimum,
                    final
                )

                pq.push(child)


initial = [ [ 1, 2, 3 ],
            [ 5, 6, 0 ],
            [ 7, 8, 4 ] ]

final = [ [ 1, 2, 3 ],
          [ 5, 8, 6 ],
          [ 0, 7, 4 ] ]

empty_tile_posi = [1, 2]

solve(initial, empty_tile_posi, final)

1  2  3  
5  6  0  
7  8  4  

1  2  3  
5  0  6  
7  8  4  

1  2  3  
5  8  6  
7  0  4  

1  2  3  
5  8  6  
0  7  4  



In [ ]:
from collections import deque

class Graph:
    def __init__(self, adjac_lis):
        self.adjac_lis = adjac_lis

    def get_neighbors(self, v):
        return self.adjac_lis[v]


    def h(self, n):
        H = {
            'A': 1,
            'B': 1,
            'C': 1,
            'D': 1
        }
        return H[n]

    def a_star_algorithm(self, start, stop):

        open_lst = set([start])
        closed_lst = set([])


        poo = {}
        poo[start] = 0


        par = {}
        par[start] = start

        while len(open_lst) > 0:
            n = None


            for v in open_lst:
                if n == None or poo[v] + self.h(v) < poo[n] + self.h(n):
                    n = v

            if n == None:
                print('Path does not exist!')
                return None


            if n == stop:
                reconst_path = []

                while par[n] != n:
                    reconst_path.append(n)
                    n = par[n]

                reconst_path.append(start)
                reconst_path.reverse()

                print('Path found: {}'.format(reconst_path))
                return reconst_path


            for (m, weight) in self.get_neighbors(n):

                if m not in open_lst and m not in closed_lst:
                    open_lst.add(m)
                    par[m] = n
                    poo[m] = poo[n] + weight
                else:

                    if poo[m] > poo[n] + weight:
                        poo[m] = poo[n] + weight
                        par[m] = n


                        if m in closed_lst:
                            closed_lst.remove(m)
                            open_lst.add(m)


            open_lst.remove(n)
            closed_lst.add(n)

        print('Path does not exist!')
        return None

adjac_lis = {
    'A': [('B', 1), ('C', 3), ('D', 7)],
    'B': [('C', 1), ('D', 5)],
    'C': [('D', 12)]
}

graph1 = Graph(adjac_lis)
graph1.a_star_algorithm('A', 'C')

Path found: ['A', 'B', 'C']


['A', 'B', 'C']

In [ ]:
#Bài 1. Cài thuật giải AKT vào bài toán 15 puzzle(n=4) code theo cách khác
import heapq
import copy

n = 4

rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]



class Node:
    def __init__(self, parent, mat, empty_pos, g, h):
        self.parent = parent
        self.mat = mat
        self.empty_pos = empty_pos
        self.g = g  # cost từ đầu
        self.h = h  # heuristic
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f



def calculate_h(mat, goal):
    dist = 0
    for i in range(n):
        for j in range(n):
            val = mat[i][j]
            if val != 0:
                for x in range(n):
                    for y in range(n):
                        if goal[x][y] == val:
                            dist += abs(x - i) + abs(y - j)
    return dist


def new_node(mat, empty_pos, new_pos, g, parent, goal):
    new_mat = copy.deepcopy(mat)

    x1, y1 = empty_pos
    x2, y2 = new_pos

    new_mat[x1][y1], new_mat[x2][y2] = new_mat[x2][y2], new_mat[x1][y1]

    h = calculate_h(new_mat, goal)

    return Node(parent, new_mat, new_pos, g, h)



def print_path(node):
    if node is None:
        return
    print_path(node.parent)
    for row in node.mat:
        print(row)
    print()



def solve(initial, empty_pos, goal):
    open_list = []
    visited = set()

    root = Node(None, initial, empty_pos, 0, calculate_h(initial, goal))
    heapq.heappush(open_list, root)

    while open_list:
        current = heapq.heappop(open_list)

        if current.mat == goal:
            print("Đã tìm thấy lời giải!\n")
            print_path(current)
            return

        visited.add(str(current.mat))

        x, y = current.empty_pos

        for i in range(4):
            new_x = x + rows[i]
            new_y = y + cols[i]

            if 0 <= new_x < n and 0 <= new_y < n:
                new_mat = copy.deepcopy(current.mat)
                new_mat[x][y], new_mat[new_x][new_y] = new_mat[new_x][new_y], new_mat[x][y]

                if str(new_mat) not in visited:
                    child = new_node(current.mat, (x, y), (new_x, new_y),
                                     current.g + 1, current, goal)
                    heapq.heappush(open_list, child)

    print("Không tìm thấy lời giải!")



initial = [
    [1, 2, 3, 4],
    [5, 6, 0, 8],
    [9, 10, 7, 11],
    [13, 14, 15, 12]
]

goal = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 0]
]

empty_pos = (1, 2)  # vị trí số 0

solve(initial, empty_pos, goal)

Đã tìm thấy lời giải!

[1, 2, 3, 4]
[5, 6, 0, 8]
[9, 10, 7, 11]
[13, 14, 15, 12]

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 0, 11]
[13, 14, 15, 12]

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 0]
[13, 14, 15, 12]

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]



In [ ]:
#Bài 2. cài đặt thuật giải A* tìm đường đi giữa 2 đỉnh bất kỳ trong đồ thị code theo cách khác
import heapq


def a_star(graph, heuristic, start, goal):
    open_list = []
    heapq.heappush(open_list, (0, start))

    g = {start: 0}
    parent = {start: None}

    while open_list:
        f, current = heapq.heappop(open_list)


        if current == goal:
            path = []
            while current:
                path.append(current)
                current = parent[current]
            return path[::-1]


        for neighbor, cost in graph[current]:
            new_g = g[current] + cost

            if neighbor not in g or new_g < g[neighbor]:
                g[neighbor] = new_g
                f = new_g + heuristic[neighbor]
                heapq.heappush(open_list, (f, neighbor))
                parent[neighbor] = current

    return None



graph = {
    'A': [('B', 1), ('C', 4)],
    'B': [('D', 2), ('E', 5)],
    'C': [('F', 3)],
    'D': [('G', 1)],
    'E': [('G', 2)],
    'F': [('G', 2)],
    'G': []
}


heuristic = {
    'A': 7,
    'B': 6,
    'C': 4,
    'D': 2,
    'E': 3,
    'F': 2,
    'G': 0
}

start = 'A'
goal = 'G'

path = a_star(graph, heuristic, start, goal)

if path:
    print("Đường đi:", " -> ".join(path))
else:
    print("Không tìm thấy đường đi")


Đường đi: A -> B -> D -> G
